In [ ]:
from pystac.client import Client
from odc.stac import load
import xarray as xr
from utils import mask_land, mask_deeps, make_indices, do_prediction
import joblib
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
collection = "sentinel-2-l2a"

In [ ]:
# # Nadi
# bbox = (177.20, -17.85, 177.50, -17.65)

# Bounding box for Tuvalu
bbox = [179.020, -8.665, 179.218, -8.413]
daterange = "2024-07/2024-12"

# # Bounding box for Suva
# bbox = [178.400, -18.200, 178.600, -18.000]

In [ ]:
# Consider removing very cloudy scenes...
items = catalog.search(
    collections=[collection], bbox=bbox, datetime=daterange
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(
    items,
    bbox=bbox,
    epsg="utm",
    measurements=[
        "scl",
        "nir",
        "red",
        "blue",
        "green",
        "nir08",
        "nir09",
        "swir16",
        "swir22",
        "coastal",
        "rededge1",
        "rededge2",
        "rededge3",
    ],
    chunks={"x": 2048, "y": 2048},
    nodata=0,
    groupby="solar_day",
)

# Mask clouds
mask = data.scl.isin([3, 8, 9, 10])
data = data.where(~mask)
data = make_indices(data)

# Mask land
data = mask_land(data)

# # Mask deep water
# data = mask_deeps(data)  # Can use threshold=1.575

data = data.drop_vars("scl")

data

In [ ]:
# Preview
# data.isel(time=0).odc.explore(bands=["red", "green", "blue"], vmin=0, vmax=2000)

In [ ]:
# # Pre-load data
# data = data.compute()

In [ ]:
model = joblib.load("models/2025_03_06_randomforest_all_no_land.joblib")

predictions_list = []

for day in tqdm(data.time):
    prediction = do_prediction(data.sel(time=day), model).compute()
    predictions_list.append(prediction)

# Concatenate them all together again
predictions = xr.concat(predictions_list, dim="time").to_dataset(name="elevation")

predictions

In [ ]:
predictions.elevation.plot.imshow(col="time", col_wrap=2, cmap="Blues_r", robust=True, size=6)

In [ ]:
from odc.algo import mask_cleanup

count = predictions.elevation.count(dim="time")
total = len(predictions.time)

# mask = count > (total * 0.35)  # At least 50% of the time there was a prediction

# mask = mask_cleanup(mask, [["erosion", 4], ["dilation", 6]])

mean = predictions.elevation.mean(dim="time")
# mean = mean.where(mask)

mean.odc.explore(cmap="Blues_r")

In [ ]:
mean.odc.write_cog("predicted_mean.tif", overwrite=True)